
# Q1 – University Management System (OOP)  
**NIBM / MSc Data Science – Programming for Data Science (NIB7143CEM)**  

This notebook contains:
- Core OOP classes (`Person`, `Student`, `Faculty`, `Department`, `Course`)  
- `SecureStudentRecord` with encapsulation + validation  
- Demo cells that produce outputs for screenshots  
- Unit tests for key functions (GPA, status, enrollment, validation)  


## Base Class: `Person` (+ `Staff`)

In [ ]:

class Person:
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email

    def get_responsibilities(self):
        return "General university member responsibilities."

    def __str__(self):
        return f"{self.name} ({self.age}) - {self.email}"


class Staff(Person):
    def __init__(self, name, age, email, role):
        super().__init__(name, age, email)
        self.role = role

    def get_responsibilities(self):
        return f"Staff role: {self.role} - Handles administrative tasks."


## Students (`Student`, `UndergraduateStudent`, `GraduateStudent`) + Encapsulated `SecureStudentRecord`

In [ ]:

class Student(Person):
    def __init__(self, name, age, email, student_id):
        super().__init__(name, age, email)
        self.student_id = student_id
        self.courses = []
        self.grades = {}

    def enroll_course(self, course):
        if course not in self.courses:
            self.courses.append(course)
            print(f"{self.name} enrolled in {course}")
        else:
            print(f"{self.name} is already enrolled in {course}")

    def drop_course(self, course):
        if course in self.courses:
            self.courses.remove(course)
            print(f"{self.name} dropped {course}")
        else:
            print(f"{self.name} is not enrolled in {course}")

    def add_grade(self, course, grade):
        if course in self.courses:
            self.grades[course] = grade
        else:
            print(f"{self.name} is not enrolled in {course}, cannot assign grade.")

    def calculate_gpa(self):
        if not self.grades:
            return 0.0
        return round(sum(self.grades.values()) / len(self.grades), 2)

    def get_academic_status(self):
        gpa = self.calculate_gpa()
        if gpa >= 3.5:
            return "Dean's List"
        elif gpa >= 2.0:
            return "Good Standing"
        else:
            return "Probation"


class UndergraduateStudent(Student):
    def get_responsibilities(self):
        return "Attend lectures, complete assignments, and exams."


class GraduateStudent(Student):
    def get_responsibilities(self):
        return "Conduct research, attend seminars, and publish papers."


class SecureStudentRecord:
    """Encapsulated student record with validation."""
    def __init__(self, student_id, gpa=0.0, max_courses=5):
        self.__student_id = student_id
        self.__gpa = 0.0
        self.__courses = []
        self.max_courses = max_courses
        self.set_gpa(gpa)  # validate on init

    # student_id (read-only getter)
    def get_student_id(self):
        return self.__student_id

    # GPA getter/setter with validation
    def get_gpa(self):
        return self.__gpa

    def set_gpa(self, value):
        if 0.0 <= value <= 4.0:
            self.__gpa = value
        else:
            raise ValueError("GPA must be between 0.0 and 4.0")

    # Course management with integrity checks
    def enroll_course(self, course):
        if len(self.__courses) >= self.max_courses:
            print(f"Enrollment failed: max limit of {self.max_courses} courses reached.")
            return False
        if course in self.__courses:
            print(f"Already enrolled in {course}.")
            return False
        self.__courses.append(course)
        print(f"Enrolled in {course}.")
        return True

    def drop_course(self, course):
        if course in self.__courses:
            self.__courses.remove(course)
            print(f"Dropped {course}.")
            return True
        else:
            print(f"Not enrolled in {course}.")
            return False

    def get_courses(self):
        return list(self.__courses)  # return a copy

    def __str__(self):
        return f"SecureStudentRecord(Student ID: {self.__student_id}, GPA: {self.__gpa}, Courses: {len(self.__courses)})"


## Faculty (`Faculty`, `Professor`, `Lecturer`, `TeachingAssistant`)

In [ ]:

class Faculty(Person):
    def __init__(self, name, age, email, faculty_id):
        super().__init__(name, age, email)
        self.faculty_id = faculty_id

    def calculate_workload(self):
        return "General teaching and administrative duties."


class Professor(Faculty):
    def get_responsibilities(self):
        return "Teach courses, conduct research, and supervise PhD students."

    def calculate_workload(self):
        return "Workload: 3 courses + research supervision."


class Lecturer(Faculty):
    def get_responsibilities(self):
        return "Teach undergraduate and graduate courses."

    def calculate_workload(self):
        return "Workload: 4 courses."


class TeachingAssistant(Faculty):
    def get_responsibilities(self):
        return "Assist faculty in grading, labs, and tutorials."

    def calculate_workload(self):
        return "Workload: 2 courses + lab sessions."


## Department & Course Management

In [ ]:

class Course:
    def __init__(self, code, name, credits, max_students=30, prerequisites=None):
        self.code = code
        self.name = name
        self.credits = credits
        self.max_students = max_students
        self.prerequisites = prerequisites if prerequisites else []
        self.enrolled_students = []  # store student *names* for demo

    def enroll_student(self, student):
        if len(self.enrolled_students) >= self.max_students:
            print(f"Course {self.name} is full.")
            return False
        if not all(pr in student.courses for pr in self.prerequisites):
            print(f"{student.name} has not completed prerequisites for {self.name}.")
            return False
        self.enrolled_students.append(student.name)
        student.enroll_course(self.name)
        return True

    def __str__(self):
        return f"{self.code} - {self.name} ({self.credits} credits)"


class Department:
    def __init__(self, name):
        self.name = name
        self.courses = []
        self.faculty = []

    def add_course(self, course):
        self.courses.append(course)

    def assign_faculty(self, faculty):
        self.faculty.append(faculty)

    def __str__(self):
        return f"Department of {self.name} with {len(self.courses)} courses."


## Demo: Create Objects, Enroll Students, Calculate GPA, Show Polymorphism

In [ ]:

# Create Department and Courses
cs_dept = Department("Computer Science")
algo = Course("CS101", "Algorithms", 3)
ml = Course("CS201", "Machine Learning", 3, prerequisites=["Algorithms"])
cs_dept.add_course(algo)
cs_dept.add_course(ml)

# Faculty
prof = Professor("Dr. Smith", 50, "smith@uni.edu", "F001")
lecturer = Lecturer("Dr. John", 40, "john@uni.edu", "F002")
ta = TeachingAssistant("Alice", 28, "alice@uni.edu", "F003")
cs_dept.assign_faculty(prof)
cs_dept.assign_faculty(lecturer)
cs_dept.assign_faculty(ta)

# Students
ug_student = UndergraduateStudent("Bob", 20, "bob@student.uni.edu", "S001")
grad_student = GraduateStudent("Eve", 24, "eve@student.uni.edu", "S002")

# Enrollments + GPA
algo.enroll_student(ug_student)
ug_student.add_grade("Algorithms", 3.8)
print(ug_student.name, "GPA:", ug_student.calculate_gpa(), "-", ug_student.get_academic_status())

ml.enroll_student(grad_student)  # should fail (no prereq)
algo.enroll_student(grad_student)
grad_student.add_grade("Algorithms", 3.2)
ml.enroll_student(grad_student)  # now should work

# Faculty responsibilities & workloads (polymorphism)
for f in [prof, lecturer, ta]:
    print(f"{f.name} responsibilities: {f.get_responsibilities()} - {f.calculate_workload()}")

# Secure record demo
record = SecureStudentRecord("S1001", gpa=3.2)
print(record)
record.set_gpa(3.9)
print("Updated GPA:", record.get_gpa())
for c in ["Algorithms", "Machine Learning", "Databases", "Statistics", "AI", "Deep Learning"]:
    record.enroll_course(c)
record.drop_course("Statistics")
print("Courses after drop:", record.get_courses())


## Unit Tests for Key Functions

In [ ]:

import unittest

class TestStudentAndRecords(unittest.TestCase):
    def test_gpa_calculation(self):
        s = Student("Test Student", 21, "test@student.uni.edu", "S100")
        s.enroll_course("Algorithms")
        s.add_grade("Algorithms", 4.0)
        self.assertEqual(s.calculate_gpa(), 4.0)

    def test_academic_status_probation(self):
        s = Student("Low GPA", 22, "low@student.uni.edu", "S101")
        s.enroll_course("Math")
        s.add_grade("Math", 1.5)
        self.assertEqual(s.get_academic_status(), "Probation")

    def test_enrollment_duplicate(self):
        s = Student("Enroll Test", 23, "enroll@uni.edu", "S102")
        s.enroll_course("Databases")
        s.enroll_course("Databases")  # duplicate
        self.assertIn("Databases", s.courses)
        self.assertEqual(s.courses.count("Databases"), 1)

    def test_secure_record_gpa_validation(self):
        rec = SecureStudentRecord("S200")
        with self.assertRaises(ValueError):
            rec.set_gpa(5.0)  # invalid

    def test_secure_record_course_limit(self):
        rec = SecureStudentRecord("S201", max_courses=2)
        self.assertTrue(rec.enroll_course("A"))
        self.assertTrue(rec.enroll_course("B"))
        self.assertFalse(rec.enroll_course("C"))  # should fail

# Run tests in notebook
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestStudentAndRecords))
